In [11]:
import yfinance as yf

ticker = "AAPL"

stock = yf.Ticker("AAPL")

income_statement = stock.financials
balance_sheet = stock.balance_sheet
cash_flow = stock.cashflow

print(income_statement)
print(balance_sheet)
print(cash_flow)

                                                      2025-09-30  \
Tax Effect Of Unusual Items                         0.000000e+00   
Tax Rate For Calcs                                  1.560000e-01   
Normalized EBITDA                                   1.447480e+11   
Net Income From Continuing Operation Net Minori...  1.120100e+11   
Reconciled Depreciation                             1.169800e+10   
Reconciled Cost Of Revenue                          2.209600e+11   
EBITDA                                              1.447480e+11   
EBIT                                                1.330500e+11   
Net Interest Income                                          NaN   
Interest Expense                                             NaN   
Interest Income                                              NaN   
Normalized Income                                   1.120100e+11   
Net Income From Continuing And Discontinued Ope...  1.120100e+11   
Total Expenses                                  

In [12]:
revenue = income_statement.loc['Total Revenue', '2025-09-30']
ebit = income_statement.loc['EBIT', '2025-09-30']
tax_provision = income_statement.loc['Tax Provision', '2025-09-30']
pretax_income = income_statement.loc['Pretax Income', '2025-09-30']
da = cash_flow.loc['Depreciation And Amortization', '2025-09-30']
capital_expenditure = cash_flow.loc['Capital Expenditure', '2025-09-30']
net_debt = balance_sheet.loc['Net Debt', '2025-09-30']
diluted_average_shares = income_statement.loc['Diluted Average Shares', '2025-09-30']

print(revenue)
print(ebit)
print(tax_provision)
print(pretax_income)
print(da)
print(capital_expenditure)
print(net_debt)
print(diluted_average_shares)


416161000000.0
133050000000.0
20719000000.0
132729000000.0
11698000000.0
-12715000000.0
62723000000.0
15004697000.0


In [17]:
effective_tax_rate = tax_provision / pretax_income
unlevered_cash_flow = ebit*(1-effective_tax_rate) + da - capital_expenditure

print(effective_tax_rate)
print(unlevered_cash_flow)

0.15610002335586043
136693891892.50276


In [18]:
growth_rate = 0.08
ufcf_margin = 0.32

projected_ufcfs = []
current_rev = revenue

for year in range(1, 6):
    current_rev = current_rev * (1 + growth_rate)
    ufcf = current_rev * ufcf_margin
    projected_ufcfs.append(ufcf)

print(projected_ufcfs)

[np.float64(143825241600.0), np.float64(155331260928.00003), np.float64(167757761802.24005), np.float64(181178382746.41925), np.float64(195672653366.1328)]


In [19]:
wacc = 0.09
discounted_ufcfs = []

for year, cf in enumerate(projected_ufcfs, 1):
    pv = cf / (1 + wacc) ** year
    discounted_ufcfs.append(pv)

print(sum(discounted_ufcfs))

647753882803.9631


In [20]:
terminal_growth_rate = 0.025

terminal_value = projected_ufcfs[-1] * (1 + terminal_growth_rate) / (wacc - terminal_growth_rate)
pv_terminal_value = terminal_value / (1 + wacc) ** 5

print(terminal_value)
print(pv_terminal_value)

3085607226158.2476
2005432982069.2217


In [21]:
enterprise_value = sum(discounted_ufcfs) + pv_terminal_value

equity_value = enterprise_value - net_debt
price_per_share = equity_value / diluted_average_shares

print(enterprise_value)
print(equity_value)
print(price_per_share)

2653186864873.1846
2590463864873.1846
172.6435305473469


In [31]:
def suggest_assumptions(ticker):
    stock = yf.Ticker(ticker)
    info = stock.info
    
    sector = info.get('sector', 'Unknown')
    revenue_growth = info.get('revenueGrowth', 0.05)
    profit_margin = info.get('profitMargins', 0.15)
    beta = info.get('beta', 1.0)
    
    risk_free = 0.045
    market_premium = 0.055
    cost_of_equity = risk_free + beta * market_premium
    estimated_wacc = round(cost_of_equity, 2)
    
    if sector in ['Technology', 'Communication Services']:
        terminal_growth = 0.03
    elif sector in ['Consumer Defensive', 'Utilities']:
        terminal_growth = 0.02
    else:
        terminal_growth = 0.025
    
    print(f"\n{'='*50}")
    print(f"SUGGESTED ASSUMPTIONS FOR {ticker}")
    print(f"{'='*50}")
    print(f"Sector: {sector}")
    print(f"Recent Revenue Growth: {revenue_growth:.1%}")
    print(f"Profit Margin: {profit_margin:.1%}")
    print(f"Beta: {beta}")
    print(f"{'='*50}")
    print(f"Suggested Growth Rate: {revenue_growth:.1%}")
    print(f"Suggested UFCF Margin: {profit_margin * 1.2:.1%}  (margin x 1.2 to approximate UFCF)")
    print(f"Suggested WACC: {estimated_wacc:.1%}")
    print(f"Suggested Terminal Growth: {terminal_growth:.1%}")
    print(f"{'='*50}\n")
    
    return revenue_growth, profit_margin * 1.2, estimated_wacc, terminal_growth


def run_dcf(ticker, growth_rate, ufcf_margin, wacc, terminal_growth_rate):

    print(f"\n{'='*50}")
    print(f"DCF VALUATION: {ticker}")
    print(f"{'='*50}")
    print(f"Year 1 Growth: {growth_rate:.0%}  (fades to terminal)")
    print(f"UFCF Margin: {ufcf_margin:.0%}")
    print(f"WACC: {wacc:.0%}")
    print(f"Terminal Growth Rate: {terminal_growth_rate:.0%}")
    print(f"{'='*50}")
    
    stock = yf.Ticker(ticker)
    income_statement = stock.financials
    balance_sheet = stock.balance_sheet
    cash_flow = stock.cashflow
    
    date = income_statement.columns[0]
    
    revenue = income_statement.loc['Total Revenue', date]
    ebit = income_statement.loc['EBIT', date]
    tax_provision = income_statement.loc['Tax Provision', date]
    pretax_income = income_statement.loc['Pretax Income', date]
    da = cash_flow.loc['Depreciation And Amortization', date]
    capital_expenditure = cash_flow.loc['Capital Expenditure', date]
    
    try:
        net_debt = balance_sheet.loc['Net Debt', date]
        if pd.isna(net_debt):
            net_debt = balance_sheet.loc['Total Debt', date] - balance_sheet.loc['Cash And Cash Equivalents', date]
    except:
        net_debt = balance_sheet.loc['Total Debt', date] - balance_sheet.loc['Cash And Cash Equivalents', date]
    
    diluted_average_shares = income_statement.loc['Diluted Average Shares', date]
    
    effective_tax_rate = tax_provision / pretax_income
    
    print(f"\nPROJECTIONS:")
    projected_ufcfs = []
    current_rev = revenue
    for year in range(1, 6):
        year_growth = growth_rate - (growth_rate - terminal_growth_rate) * (year - 1) / 4
        current_rev = current_rev * (1 + year_growth)
        projected_ufcf = current_rev * ufcf_margin
        projected_ufcfs.append(projected_ufcf)
        print(f"Year {year}: Growth {year_growth:.1%} | Revenue ${current_rev/1e9:,.1f}B | UFCF ${projected_ufcf/1e9:,.1f}B")
    
    discounted_ufcfs = []
    for year, cf in enumerate(projected_ufcfs, 1):
        pv = cf / (1 + wacc) ** year
        discounted_ufcfs.append(pv)
    
    terminal_value = projected_ufcfs[-1] * (1 + terminal_growth_rate) / (wacc - terminal_growth_rate)
    pv_terminal_value = terminal_value / (1 + wacc) ** 5
    
    enterprise_value = sum(discounted_ufcfs) + pv_terminal_value
    equity_value = enterprise_value - net_debt
    price_per_share = equity_value / diluted_average_shares
    
    print(f"\n{'='*50}")
    print(f"VALUATION OUTPUT")
    print(f"{'='*50}")
    print(f"Ticker: {ticker}")
    print(f"Implied Price Per Share: ${price_per_share:,.2f}")
    print(f"Enterprise Value: ${enterprise_value/1e9:,.1f}B")
    print(f"Equity Value: ${equity_value/1e9:,.1f}B")
    print(f"{'='*50}")


g, m, w, t = suggest_assumptions('NVDA')
run_dcf('NVDA', g, m, w, t)


SUGGESTED ASSUMPTIONS FOR NVDA
Sector: Technology
Recent Revenue Growth: 85.2%
Profit Margin: 63.0%
Beta: 2.202
Suggested Growth Rate: 85.2%
Suggested UFCF Margin: 75.6%  (margin x 1.2 to approximate UFCF)
Suggested WACC: 17.0%
Suggested Terminal Growth: 3.0%


DCF VALUATION: NVDA
Year 1 Growth: 85%  (fades to terminal)
UFCF Margin: 76%
WACC: 17%
Terminal Growth Rate: 3%

PROJECTIONS:
Year 1: Growth 85.2% | Revenue $399.9B | UFCF $302.2B
Year 2: Growth 64.6% | Revenue $658.5B | UFCF $497.5B
Year 3: Growth 44.1% | Revenue $948.8B | UFCF $716.9B
Year 4: Growth 23.6% | Revenue $1,172.3B | UFCF $885.8B
Year 5: Growth 3.0% | Revenue $1,207.5B | UFCF $912.4B

VALUATION OUTPUT
Ticker: NVDA
Implied Price Per Share: $204.75
Enterprise Value: $5,019.7B
Equity Value: $5,019.3B


In [33]:
g, m, w, t = suggest_assumptions('AAPL')
run_dcf('AAPL', g, m, w, t)


SUGGESTED ASSUMPTIONS FOR AAPL
Sector: Technology
Recent Revenue Growth: 16.6%
Profit Margin: 27.2%
Beta: 1.086
Suggested Growth Rate: 16.6%
Suggested UFCF Margin: 32.6%  (margin x 1.2 to approximate UFCF)
Suggested WACC: 10.0%
Suggested Terminal Growth: 3.0%


DCF VALUATION: AAPL
Year 1 Growth: 17%  (fades to terminal)
UFCF Margin: 33%
WACC: 10%
Terminal Growth Rate: 3%

PROJECTIONS:
Year 1: Growth 16.6% | Revenue $485.2B | UFCF $158.1B
Year 2: Growth 13.2% | Revenue $549.3B | UFCF $179.0B
Year 3: Growth 9.8% | Revenue $603.1B | UFCF $196.5B
Year 4: Growth 6.4% | Revenue $641.7B | UFCF $209.1B
Year 5: Growth 3.0% | Revenue $661.0B | UFCF $215.4B

VALUATION OUTPUT
Ticker: AAPL
Implied Price Per Share: $174.66
Enterprise Value: $2,683.5B
Equity Value: $2,620.7B
